In [ ]:
import re
import os
import subprocess
import pandas as pd
import numpy as np
from pathlib import PurePath, PurePosixPath
from h_anonypy.modules_video import compute_sha256
from h_anonypy.modules_video import get_video_infomation, check_patient_ids, is_split_screen, check_split_screen
from h_anonypy.modules_video import get_video_metadata_ffprobe, get_video_metadata_opencv, capture_key_frames_by_video
import shutil


In [3]:
os.name


'posix'

In [ ]:
##### INPUT #####
# IMAGE_META : 전체 영상 메타정보 
# VIDEO_META : 전체 비디오 메타정보
# HUTOM_ID : 전체 hutom id 리스트
# video_dir : 반입 데이터 경로
# center: 반입 기관
# importdate: 반입 날짜
# organ: 조직 정보
# n_digits: hutom id 자리수

# split_str : 원본 파일명으로부터 새 파일명 정보 추출을 위한 파라미터
# select_n : 원본 파일명으로부터 새 파일명 정보 추출을 위한 파라미터

IMAGE_META = pd.read_excel('./SHEET/IMAGE_META.xlsx', sheet_name=None)
VIDEO_META = pd.read_excel('./SHEET/VIDEO_META.xlsx', sheet_name=None)
HUTOM_ID = pd.read_excel('./SHEET/HUTOM_ID.xlsx', sheet_name=None)
video_dir = ['/nas/nas6/DataTeam/URO/[VIDEO]화순전남대병원-황의창_20251202_10건']
center = ['JNUH']
importdate = ['20251202']
organ = ['URO']
n_digits = 4

# check ID
split_str = ' '
select_n = 1


In [ ]:
## Start [ver.2025.08]
for i in range(len(video_dir)):
    # 1. 경로 내 비디오 파일 해시 추출
    source_info = get_video_infomation(video_dir[i])

    # 2. patient_id 수정 [여러 비디오가 존재하는 경우]
    posix = PurePath(video_dir[i])
    video_info = check_patient_ids(source_info, video_dir[i], id_path_index=len(posix.parts))
    video_info.insert(0, 'hutom_id', None)

    # 3. 메타정보 추출, 중복 제거 및 HUTOM ID 부여
    meta_all = VIDEO_META[organ[i]].copy()
    image_meta_all = IMAGE_META[organ[i]].copy()
    ids_all = HUTOM_ID[organ[i]].copy()
    mask = ~ids_all['hutom_id'].astype(str).str.contains('FDA', case=False, na=False)
    hutom_ids = ids_all.loc[mask,'hutom_id'].dropna().unique().tolist()
    nums = [int(m.group(1)) for s in hutom_ids if (m := re.search(r'(\d+)$', s))]
    id_number = np.sort(nums)[-1] + 1

    save_dir = os.path.join(video_dir[i],'capture') 
    video_info[['size(bytes)','width','height','codec_name','fps','nb_frames','duration']] = None
    ids = video_info['patient_id'].unique().tolist()
    for j in range(len(ids)):
        # 3.1 메타 정보 추출
        
        check_sample = video_info[video_info['patient_id'] == ids[j]]
        check_idx = check_sample.index.tolist()

        idx_ch = 0
        idx_cont = 1  
        idx_split = 1 
        for k in range(len(check_idx)):
            filepath = check_sample.loc[check_idx[k],'filepath']
            meta_ffprobe = get_video_metadata_ffprobe(filepath)
            if not meta_ffprobe: 
                video_info.loc[check_idx[k],'format'] = 'dameged_file'
                continue 
            meta_opencv = get_video_metadata_opencv(filepath)
            video_stream = meta_ffprobe['streams'][0]
            video_info.loc[check_idx[k],'size(bytes)'] = os.path.getsize(filepath)
            video_info.loc[check_idx[k],'width'] = video_stream.get('width', meta_opencv['width'])
            video_info.loc[check_idx[k],'height'] = video_stream.get('height', meta_opencv['height'])
            video_info.loc[check_idx[k],'codec_name'] = video_stream.get('codec_name')
            try:
                video_info.loc[check_idx[k],'fps'] = eval(video_stream.get('avg_frame_rate'))
            except:
                video_info.loc[check_idx[k],'fps'] = meta_opencv['fps']
            video_info.loc[check_idx[k],'nb_frames'] = video_stream.get('nb_frames', meta_opencv['nb_frames'])
            video_info.loc[check_idx[k],'duration'] = float(video_stream.get('duration', meta_opencv['duration']))

            # 캡처 이미지 및 채널명 생성
            frames = capture_key_frames_by_video(filepath, video_dir[i], save_dir)
            # check split image
            name, ext = os.path.splitext(os.path.basename(filepath))
            check_vertical, check_horizontal = is_split_screen(frames[1])
            if len(check_vertical) == 1 and len(check_horizontal) == 1:
                if 'ch' in name:
                    channel = re.search(r"(ch\d+)", name).group(1)
                    if idx_cont > 1:
                        if reset_channel != channel:
                            idx_cont = 1
                    ch_name = f"{channel}_{idx_cont:02d}"
                else:
                    ch_name = f"ch{idx_ch}_{idx_cont:02d}" # ch0_01
                idx_cont +=1
                reset_channel = channel
            else:
                ch_name = f"split_{idx_split:02d}" # split_01, split_02
                idx_split += 1
            video_info.loc[check_idx[k],'ch_name'] = ch_name
            
        # 3.2. ID 재구성 > 확인이 되는 경우 재추출
        if split_str is not None:
            id_re = ids[j].split(split_str)[select_n] 
            video_info.loc[check_idx, "patient_id"] = id_re
        
        # 3.3. 휴톰 아이디 생성
        hash_list = check_sample['hash'].unique().tolist()
        id_list = check_sample['patient_id'].unique().tolist()
        dup = meta_all[meta_all['Hash'].isin(hash_list)]
        dup_img = image_meta_all[image_meta_all['PatientID'].isin(id_list)]

        if len(dup)+len(dup_img) == 0:
            hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
            video_info.loc[check_idx, "hutom_id"] = hutomid
            id_number += 1
        elif len(dup)+len(dup_img) > 0:
            if len(dup) == 0:
                hutomid = dup_img["hutom_id"].tolist()[0]
            elif len(dup_img) == 0:
                hutomid = dup["hutom_id"].tolist()[0]
            video_info.loc[check_idx, "hutom_id"] = hutomid

    # 4. 익명화
    for l in range(len(video_info)):

        filepath = video_info.loc[l,'filepath']

        hutomid = video_info.loc[l,'hutom_id']
        anonyid = f"{hutomid}_{video_info.loc[l,'ch_name']}.{video_info.loc[l,'format']}"
        anony_folder = os.path.join(video_dir[i], 'ANONYMOUS', hutomid)
        anony_filename = os.path.join(anony_folder, anonyid)

        os.makedirs(anony_folder, exist_ok=True)

        if "ch" in video_info.loc[l,'ch_name']:
            anony_filename = os.path.join(video_dir[i], 'ANONYMOUS', hutomid, anonyid)
            shutil.copyfile(filepath, anony_filename)

            video_info.loc[l,'filepath'] = str(PurePath(*PurePath(filepath).parts[2:]))
            video_info.loc[l,'anony_filepath'] = str(PurePath(*PurePath(anony_filename).parts[2:]))

    # 5. Add information > check !!!
    ids_add = pd.DataFrame(video_info['hutom_id'].unique().tolist(),
                           columns=['hutom_id'])
    ids_add['video'] = 'O'
    ids_all_add = pd.merge(ids_all, ids_add, on='hutom_id', how='outer')
    ids_all_add["video"] = ids_all_add["video_x"].fillna(ids_all_add["video_y"])
    ids_all_add = ids_all_add[['hutom_id','dicom','video']]

    video_info = video_info.rename(columns={'filepath':'RAW_PATH',
                                            'anony_filepath':'SOURCE_PATH',
                                            'hash':'Hash'})
    meta_all_add = pd.concat([meta_all, 
                              video_info[meta_all.columns.tolist()]], ignore_index=True)

##### OUTPUT > VIDEO_META, HUTOM_ID 업데이트 및 저장
# VIDEO_META[organ[i]]
# meta_all_add



In [ ]:
## Start [ver.2025.11]

i=0

source_info = get_video_infomation(video_dir[i])
posix = PurePath(video_dir[i])
video_info = check_patient_ids(source_info, video_dir[i], id_path_index=len(posix.parts))
video_info.insert(0, 'hutom_id', None)

# 3. 메타정보 추출, 중복 제거 및 HUTOM ID 부여
if organ[i] in list(HUTOM_ID.keys()):
    meta_all = VIDEO_META[organ[i]].copy()
    image_meta_all = IMAGE_META[organ[i]].copy()
    ids_all = HUTOM_ID[organ[i]].copy()
    mask = ~ids_all['hutom_id'].astype(str).str.contains('FDA', case=False, na=False)
    hutom_ids = ids_all.loc[mask,'hutom_id'].dropna().unique().tolist()
    nums = [int(m.group(1)) for s in hutom_ids if (m := re.search(r'(\d+)$', s))]
    id_number = np.sort(nums)[-1] + 1
else:
    id_number = 1

save_dir = os.path.join(video_dir[i],'capture') 
video_info[['new_hash','size(bytes)','width','height','codec_name','fps','nb_frames','duration','split_info']] = None
ids = video_info['patient_id'].unique().tolist()
for j in range(len(ids)):
    # 3.1 메타 정보 추출
    
    check_sample = video_info[video_info['patient_id'] == ids[j]]
    check_idx = check_sample.index.tolist()

    idx_ch = 0
    idx_cont = 1  
    idx_split = 1 
    for k in range(len(check_idx)):
        filepath = check_sample.loc[check_idx[k],'filepath']
        meta_ffprobe = get_video_metadata_ffprobe(filepath)
        if not meta_ffprobe: 
            video_info.loc[check_idx[k],'format'] = 'dameged_file'
            continue 
        meta_opencv = get_video_metadata_opencv(filepath)
        video_stream = meta_ffprobe['streams'][0]
        video_info.loc[check_idx[k],'size(bytes)'] = os.path.getsize(filepath)
        video_info.loc[check_idx[k],'width'] = video_stream.get('width', meta_opencv['width'])
        video_info.loc[check_idx[k],'height'] = video_stream.get('height', meta_opencv['height'])
        video_info.loc[check_idx[k],'codec_name'] = video_stream.get('codec_name')
        try:
            video_info.loc[check_idx[k],'fps'] = eval(video_stream.get('avg_frame_rate'))
        except:
            video_info.loc[check_idx[k],'fps'] = meta_opencv['fps']
        video_info.loc[check_idx[k],'nb_frames'] = video_stream.get('nb_frames', meta_opencv['nb_frames'])
        video_info.loc[check_idx[k],'duration'] = float(video_stream.get('duration', meta_opencv['duration']))

        # 3.2 캡처 이미지 생성 및 분할 영상 확인
        frames = capture_key_frames_by_video(filepath, video_dir[i], save_dir)
        check_screen = check_split_screen(frames)
        video_info.loc[check_idx[k],'split_info'] = check_screen
        name, ext = os.path.splitext(os.path.basename(filepath))
        if (check_screen == 'horizontal') or (check_screen == 'vertical'):
            ch_name = f"split_{idx_split:02d}"
            idx_split += 1
        else:
            if 'ch' in name:
                ch_name = name
            else:
                ch_name = f"ch0_{idx_cont:02d}" # ch0_01
            idx_cont +=1
        video_info.loc[check_idx[k],'ch_name'] = ch_name

    # 3.3. 휴톰 아이디 생성 [중복 체크 후 아이디 생성]
    hash_list = check_sample['hash'].unique().tolist()
    id_list = check_sample['patient_id'].unique().tolist()
    if organ[i] in list(HUTOM_ID.keys()):
        dup = meta_all[meta_all['Hash'].isin(hash_list)]
        dup_img = image_meta_all[image_meta_all['PatientID'].isin(id_list)]
    else:
        dup, dup_img = [], []
    
    if len(dup)+len(dup_img) == 0:
        hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
        video_info.loc[check_idx, "hutom_id"] = hutomid
        id_number += 1
    elif len(dup)+len(dup_img) > 0:
        if len(dup) == 0:
            hutomid = dup_img["hutom_id"].tolist()[0]
        elif len(dup_img) == 0:
            hutomid = dup["hutom_id"].tolist()[0]
        video_info.loc[check_idx, "hutom_id"] = hutomid

# 4. 익명화
filepath = None
anony_filename = None
cmd = [
    "ffmpeg",
    "-i", filepath,
    "-c:v", "libx264",
    "-profile:v", "high",
    "-level", "4.0",
    "-pix_fmt", "yuv420p",
    "-s", "1280x1024",
    "-r", "30",
    "-b:v", "6962k",
    "-c:a", "aac",
    "-profile:a", "aac_low",
    "-ar", "48000",
    "-ac", "2",
    "-b:a", "128k",
    "-movflags", "+faststart",
    anony_filename
]
for l in range(len(video_info)):
    
    filepath = video_info.loc[l,'filepath']
    cmd[2] = filepath
    
    hutomid = video_info.loc[l,'hutom_id']
    anonyid = f"{hutomid}_{video_info.loc[l,'ch_name']}.mp4"
    anony_folder = os.path.join(video_dir[i], 'ANONYMOUS', hutomid)
    os.makedirs(anony_folder, exist_ok=True)
    
    # if "ch" in video_info.loc[l,'ch_name']:
    anony_filename = os.path.join(video_dir[i], 'ANONYMOUS', hutomid, anonyid)
    cmd[-1] = anony_filename
    # 실행
    # shutil.copyfile(filepath, anony_filename)
    subprocess.run(cmd, check=True)

    video_info.loc[l,'new_hash'] = compute_sha256(anony_filename) 
    video_info.loc[l,'filepath'] = str(PurePath(*PurePath(filepath).parts[2:]))
    video_info.loc[l,'anony_filepath'] = str(PurePath(*PurePath(anony_filename).parts[2:]))

# 3.4. 추출 메타 정보 저장
video_info = video_info.rename(columns={'filepath':'RAW_PATH',
                                        'anony_filepath':'SOURCE_PATH',
                                        'hash':'Hash',
                                        'new_hash':'NEW_Hash'})
video_info['Center'] = center[i]
video_info['ImportDate'] = importdate[i]
video_info.to_excel(video_dir[i] + '/video_info_' + importdate[i] + '.xlsx', index=None)

video_info



In [99]:

# 5. Add information > check !!!
ids_add = pd.DataFrame(video_info['hutom_id'].unique().tolist(),
                        columns=['hutom_id'])
ids_add['video'] = 'O'

if organ[i] in list(HUTOM_ID.keys()):
    ids_all_add = pd.merge(ids_all, ids_add, on='hutom_id', how='outer')
    ids_all_add["video"] = ids_all_add["video_x"].fillna(ids_all_add["video_y"])
    ids_all_add = ids_all_add[['hutom_id','dicom','video']]
    meta_all_add = pd.concat([meta_all, video_info[meta_all.columns.tolist()]], ignore_index=True)
else:
    ids_all_add = ids_add.copy()
    ids_all_add.insert(1,'dicom', None)
    col_list = ['hutom_id', 'Hash', 'NEW_Hash', 'RAW_PATH', 'SOURCE_PATH',
            'size(bytes)', 'width', 'height', 'codec_name', 'fps', 'nb_frames',
            'duration', 'split_info', 'ch_name', 'ImportDate']
    meta_all_add = video_info[col_list].copy()

VIDEO_META[organ[i]] = meta_all_add
with pd.ExcelWriter('./SHEET/VIDEO_META.xlsx', engine='openpyxl') as writer:
    for sheet_name, data in VIDEO_META.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)
        
HUTOM_ID[organ[i]] = ids_all_add
with pd.ExcelWriter('./SHEET/HUTOM_ID.xlsx', engine='openpyxl') as writer:
    for sheet_name, data in HUTOM_ID.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)


In [ ]:
HUTOM_ID.keys()




In [ ]:
import pandas as pd
import numpy as np
import os
import datetime as dt
from tqdm.notebook import tqdm
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt

def build_queries(qualified: str):
    """
    입력: 'schema.table' 형식의 문자열
    출력: (count_sql, select_sql)
    """
    qualified = qualified.strip().strip('"').strip("'")
    if '.' in qualified:
        schema, table = qualified.split('.', 1)
    else:
        schema, table = 'public', qualified  # 스키마 생략 시 public 가정

    # 1) 비인용(그대로) 버전
    count_sql  = f"SELECT COUNT(*) AS n FROM {schema}.{table}"

    # 2) 식별자 인용(안전) 버전
    select_sql = f'SELECT * FROM "{schema}"."{table}"'

    return count_sql, select_sql

def extract_db(engine, table_name):
    
    count_sql, select_sql = build_queries(table_name)

    # 크기 확인
    n = pd.read_sql(count_sql, engine)["n"][0]
    print("rows:", n)
    
    # Load DB
    df_head = pd.read_sql(text(select_sql), engine)
    
    return df_head

USER = 'postgres'
PWD  = 'postgres'
HOST = '192.168.16.16'      # 또는 DB 서버 주소
PORT = 6543
DB   = 'Hutom'

# URL = f'postgresql+psycopg2://{USER}:{PWD}@{HOST}:{PORT}/{DB}'
# engine = create_engine(URL, pool_pre_ping=True)
# video_info = extract_db(engine, 'public.tbl_data_import_video')
# # db_raw_lab = extract_db(engine, 'hutom_bronze.tbl_labdata')


In [ ]:
import psycopg2

# 데이터베이스 연결
conn = psycopg2.connect(
    host=HOST,      # 또는 DB 서버 주소
    dbname=DB,
    user=USER,
    password=PWD,
    port=PORT
)
# SQL 쿼리 실행
query = "SELECT * FROM public.tbl_data_import_video WHERE duration IS NULL;"
query = "SELECT * FROM public.tbl_data_import_video WHERE codec_name  = 'hevc';"
query = "SELECT * FROM public.tbl_data_import_video WHERE codec_name  = 'mpeg4';"
query = "SELECT * FROM public.tbl_data_import_video WHERE codec_name  = 'mpeg2video';"
query = "SELECT * FROM hutom_bronze.sp_hutom_id;"

# 결과를 pandas DataFrame으로 불러오기
df = pd.read_sql(query, conn)
# df['sourcedata_path'].tolist()

df.to_csv('UGI_MissingID_Mapping.csv', index=None)

# source_flist = df['sourcedata_path'].tolist()
# os.path.join('/nas/nas6/', '/'.join(source_flist[0].split('/')[3:]))


In [ ]:
from h_anonypy.modules_video import compute_sha256
from tqdm.notebook import tqdm

col_list = ['size','hash','width','height','codec_name','fps','nb_frames','duration']
source_flist = df['sourcedata_path'].tolist()

for i in tqdm(range(len(source_flist))):
    filepath = os.path.join('/nas/nas6/', '/'.join(source_flist[i].split('/')[3:]))

    df.loc[i,'hash'] = compute_sha256(filepath)
    df.loc[i,'size'] = os.path.getsize(filepath)
    meta_ffprobe = get_video_metadata_ffprobe(filepath)
    meta_opencv = get_video_metadata_opencv(filepath)
    video_stream = meta_ffprobe['streams'][0]
    df.loc[i,'width'] = video_stream.get('width', meta_opencv['width'])
    df.loc[i,'height'] = video_stream.get('height', meta_opencv['height'])
    df.loc[i,'codec_name'] = video_stream.get('codec_name')
    try:
        df.loc[i,'fps'] = eval(video_stream.get('avg_frame_rate'))
    except:
        df.loc[i,'fps'] = meta_opencv['fps']
    df.loc[i,'nb_frames'] = video_stream.get('nb_frames', meta_opencv['nb_frames'])
    df.loc[i,'duration'] = float(video_stream.get('duration', meta_opencv['duration']))

df

,hutom_id,PatientID,PatientName,PatientSex,PatientAge,PatientBirthDate,AcquisitionDate,PatientSize,PatientWeight,OtherPatientIDs,OtherPatientNames,InstitutionName,ReferringPhysicianName,AccessionNumber,Modality,BodyPartExamined,folder,Center,ImportDate
5,UGI0004,ANONYMIZE_0004,UNKNOWN,M,084Y,19380701.0,20141126,NaN,NaN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,4376031,CT,NaN,UGI\데이터서비스팀\127_1268060_20220701_CT,SEVERANCE,20250624
6,UGI0004,ANONYMIZE_0004,ANONYMIZE,F,0Y,19000101.0,20141126,NaN,NaN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,671980,CT,NaN,UGI\[CT]신촌세브란스 501\ANONYMIZE_0004\1,SEVERANCE,20240108
